# BERT Model Experiments

This notebook prepares and fine-tunes a pretrained BERT model to classify crisis-related social media posts into 11 humanitarian categories. It uses the shared preprocessed datasets from `data/processed/`, with `text_clean` as the model input and `class_label` as the target. Model performance will be monitored on the validation set, with the test set reserved for final evaluation.

In [1]:
#imports
import sys
from pathlib import Path
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Make the project root available so Python can import src.bert_model.
project_root = Path(
    "/Users/emilybartell/Documents/CS1684 Crisis NLP/cs1684-crisis-nlp"
)

if not (project_root / "src" / "bert_model.py").exists():
    raise FileNotFoundError(
        f"Could not find src/bert_model.py in {project_root}"
    )

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)


/Users/emilybartell/Documents/CS1684 Crisis NLP/cs1684-crisis-nlp/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Project root: /Users/emilybartell/Documents/CS1684 Crisis NLP/cs1684-crisis-nlp


In [2]:
from src.bert_model import load_data, prepare_labels, tokenize_data

# Load the shared, preprocessed datasets.
train_df, dev_df, test_df = load_data()

# Create a consistent numerical mapping for the 11 categories.
train_df, dev_df, test_df, label2id, id2label = prepare_labels(
    train_df, dev_df, test_df
)

print("Training examples:", f"{len(train_df):,}")
print("Validation examples:", f"{len(dev_df):,}")
print("Test examples:", f"{len(test_df):,}")
print("Number of categories:", len(label2id))

print("\nTraining label counts:")
print(train_df["class_label"].value_counts())


Training examples: 61,164
Validation examples: 8,934
Test examples: 17,356
Number of categories: 11

Training label counts:
class_label
not_humanitarian                       36071
donation_and_volunteering               5175
requests_or_needs                       4829
sympathy_and_support                    3545
infrastructure_and_utilities_damage     3539
affected_individual                     2452
caution_and_advice                      2100
injured_or_dead_people                  1942
response_efforts                         780
missing_and_found_people                 373
displaced_and_evacuations                358
Name: count, dtype: int64


## Initialize BERT

Load the pretrained BERT tokenizer and classification model. The classifier is configured for the project's 11 categories and uses Apple GPU acceleration when available.

In [3]:
#initialize the model
MODEL_NAME = "bert-base-uncased"
NUM_LABELS = len(label2id)
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
)

device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cpu"
)
model.to(device)

print("Model initialized successfully.")
print("Device:", device)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model initialized successfully.
Device: mps


### Prepare Text and Labels

BERT requires text to be converted into tokens and labels to be represented numerically. This section creates a consistent mapping for the 11 humanitarian categories and tokenizes the training, validation, and test datasets.

In [4]:
# Create a consistent mapping of category names to numeric labels
label_names = sorted(train_df["class_label"].unique())

label2id = {label: i for i, label in enumerate(label_names)}
id2label = {i: label for label, i in label2id.items()}

# Convert category names to numeric labels
train_df["label"] = train_df["class_label"].map(label2id)
dev_df["label"] = dev_df["class_label"].map(label2id)
test_df["label"] = test_df["class_label"].map(label2id)

print("Number of classes:", len(label_names))
print("\nLabel mapping:")
print(label2id)

Number of classes: 11

Label mapping:
{'affected_individual': 0, 'caution_and_advice': 1, 'displaced_and_evacuations': 2, 'donation_and_volunteering': 3, 'infrastructure_and_utilities_damage': 4, 'injured_or_dead_people': 5, 'missing_and_found_people': 6, 'not_humanitarian': 7, 'requests_or_needs': 8, 'response_efforts': 9, 'sympathy_and_support': 10}


In [5]:
#tokenize and test

# Tokenize text_clean and create PyTorch datasets.
train_dataset, dev_dataset, test_dataset = tokenize_data(
    train_df, dev_df, test_df, tokenizer
)

# Verify dataset sizes and label consistency.
assert len(train_dataset) == len(train_df)
assert len(dev_dataset) == len(dev_df)
assert len(test_dataset) == len(test_df)
assert len(label2id) == 11

for name, df in [
    ("training", train_df),
    ("validation", dev_df),
    ("test", test_df),
]:
    assert df["label"].notna().all(), f"Missing labels in {name}"
    assert set(df["class_label"]) == set(label2id), (
        f"Unexpected categories in {name}"
    )

# Check one example.
example = train_dataset[0]
assert example["input_ids"].shape[0] == MAX_LENGTH
assert example["attention_mask"].shape[0] == MAX_LENGTH

print("All tests passed!")
print(f"Training examples: {len(train_dataset):,}")
print(f"Validation examples: {len(dev_dataset):,}")
print(f"Test examples: {len(test_dataset):,}")
print(f"Number of categories: {len(label2id)}")
print("Model input: text_clean")
print("Target: class_label")
print("Device:", device)

print("\nExample input shape:", example["input_ids"].shape)
print("Example numeric label:", example["labels"].item())
print("Example category:", id2label[example["labels"].item()])


All tests passed!
Training examples: 61,164
Validation examples: 8,934
Test examples: 17,356
Number of categories: 11
Model input: text_clean
Target: class_label
Device: mps

Example input shape: torch.Size([128])
Example numeric label: 4
Example category: infrastructure_and_utilities_damage


In [6]:
import importlib
import src.bert_model

importlib.reload(src.bert_model)

from src.bert_model import train_model

In [7]:
# Train and evaluate BERT on a small sample first
model, training_history = train_model(
    train_dataset,
    dev_dataset,
    label2id,
    id2label,
)
history_df = pd.DataFrame(training_history)
display(history_df)

Training device: mps
Training examples: 61164
Validation examples: 8934


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


KeyboardInterrupt: 

In [ ]:
#save models
from pathlib import Path
import shutil

project_root = Path.cwd().parent  # if your notebook is in notebooks/
source = Path.cwd() / "models" / "bert_crisis_classifier"
destination = project_root / "models" / "bert_crisis_classifier"

destination.parent.mkdir(parents=True, exist_ok=True)

if source.resolve() != destination.resolve():
    shutil.copytree(source, destination, dirs_exist_ok=True)

print(f"Model files are at: {destination.resolve()}")

Model files are at: /Users/emilybartell/Documents/CS1684 Crisis NLP/cs1684-crisis-nlp/models/bert_crisis_classifier


In [9]:
# Generate validation predictions using the saved BERT model
import importlib
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import src.bert_model as bert_model

importlib.reload(bert_model)

# Locate the saved model
project_root = Path.cwd().parent
model_path = project_root / "models" / "bert_crisis_classifier"

# Load the tokenizer and trained model
tokenizer = AutoTokenizer.from_pretrained(model_path)

model = AutoModelForSequenceClassification.from_pretrained(model_path)

# Use the available device
device = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)

model.to(device)

print("Loaded saved BERT model from:", model_path)
print("Device:", device)

# Generate validation predictions
dev_predictions = bert_model.predict(model, dev_dataset)

# Save predictions
bert_model.save_predictions(
    dev_predictions,
    project_root / "outputs" / "bert_dev_predictions.csv"
)

print("Number of validation predictions:", len(dev_predictions))
print("Number of validation examples:", len(dev_dataset))
print("First 10 predictions:", dev_predictions[:10])

assert len(dev_predictions) == len(dev_dataset)
print("Prediction test passed!")


Loaded saved BERT model from: /Users/emilybartell/Documents/CS1684 Crisis NLP/cs1684-crisis-nlp/models/bert_crisis_classifier
Device: mps
Saved 8934 predictions to /Users/emilybartell/Documents/CS1684 Crisis NLP/cs1684-crisis-nlp/outputs/bert_dev_predictions.csv
Number of validation predictions: 8934
Number of validation examples: 8934
First 10 predictions: [7, 4, 7, 7, 3, 7, 7, 10, 3, 7]
Prediction test passed!


In [10]:
# Generate test predictions using the saved BERT model

test_predictions = bert_model.predict(model, test_dataset)

bert_model.save_predictions(
    test_predictions,
    project_root / "outputs" / "bert_test_predictions.csv"
)

print("Number of test predictions:", len(test_predictions))
print("Number of test examples:", len(test_dataset))

assert len(test_predictions) == len(test_dataset)

print("Test prediction test passed!")

Saved 17356 predictions to /Users/emilybartell/Documents/CS1684 Crisis NLP/cs1684-crisis-nlp/outputs/bert_test_predictions.csv
Number of test predictions: 17356
Number of test examples: 17356
Test prediction test passed!


In [16]:
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

# Load saved test predictions
test_predictions_df = pd.read_csv(
    project_root / "outputs" / "bert_test_predictions.csv"
)

# Extract actual and predicted labels
actual_test_labels = test_dataset.labels
predicted_test_labels = test_predictions_df["prediction"].tolist()

# Confirm matching lengths
assert len(actual_test_labels) == len(predicted_test_labels)

# Calculate metrics
accuracy = accuracy_score(actual_test_labels, predicted_test_labels)

precision, recall, f1, _ = precision_recall_fscore_support(
    actual_test_labels,
    predicted_test_labels,
    average="macro",
    zero_division=0
)

print("FINAL BERT TEST RESULTS")
print("-----------------------")
print(f"Test examples: {len(actual_test_labels)}")
print(f"Accuracy: {accuracy:.4f}")
print(f"Macro Precision: {precision:.4f}")
print(f"Macro Recall: {recall:.4f}")
print(f"Macro F1: {f1:.4f}")

print("\nCLASSIFICATION REPORT")
print(classification_report(
    actual_test_labels,
    predicted_test_labels,
    digits=4,
    zero_division=0
))

print("\nCONFUSION MATRIX")
display(pd.DataFrame(
    confusion_matrix(actual_test_labels, predicted_test_labels),
    index=[f"Actual {i}" for i in range(11)],
    columns=[f"Predicted {i}" for i in range(11)]
))

FINAL BERT TEST RESULTS
-----------------------
Test examples: 17356
Accuracy: 0.8761
Macro Precision: 0.7203
Macro Recall: 0.7476
Macro F1: 0.7333

CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0     0.7718    0.8454    0.8069       692
           1     0.7005    0.7461    0.7226       583
           2     0.5849    0.6263    0.6049        99
           3     0.7940    0.8258    0.8096      1452
           4     0.7227    0.7255    0.7241      1002
           5     0.8224    0.8913    0.8554       561
           6     0.4762    0.4854    0.4808       103
           7     0.9477    0.9252    0.9363     10253
           8     0.9296    0.9255    0.9276      1370
           9     0.3913    0.4480    0.4177       221
          10     0.7825    0.7794    0.7809      1020

    accuracy                         0.8761     17356
   macro avg     0.7203    0.7476    0.7333     17356
weighted avg     0.8794    0.8761    0.8775     17356


CONFUSION MATRI

,Predicted 0,Predicted 1,Predicted 2,Predicted 3,Predicted 4,Predicted 5,Predicted 6,Predicted 7,Predicted 8,Predicted 9,Predicted 10
Actual 0,585,13,0,22,24,3,1,26,0,0,18
Actual 1,10,435,3,28,30,4,3,51,3,1,15
Actual 2,1,3,62,6,3,3,5,8,1,4,3
Actual 3,39,15,12,1199,27,5,9,72,8,42,24
Actual 4,42,45,1,29,727,19,2,79,34,3,21
Actual 5,6,5,4,8,8,500,5,22,0,0,3
Actual 6,2,1,9,11,2,6,50,14,2,3,3
Actual 7,49,84,10,144,146,53,16,9486,46,89,130
Actual 8,1,2,2,9,22,4,7,43,1268,11,1
Actual 9,0,1,2,18,2,7,6,83,0,99,3


In [17]:
from pathlib import Path
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

# Create an evaluation results folder
results_dir = project_root / "outputs" / "bert_evaluation"
results_dir.mkdir(parents=True, exist_ok=True)

# Get the category names
class_names = [
    id2label[i] if i in id2label
    else id2label[str(i)] if str(i) in id2label
    else f"Class {i}"
    for i in range(11)
]

# Save overall metrics
metrics_df = pd.DataFrame([{
    "test_examples": len(actual_test_labels),
    "accuracy": accuracy,
    "macro_precision": precision,
    "macro_recall": recall,
    "macro_f1": f1
}])

metrics_df.to_csv(results_dir / "bert_test_metrics.csv", index=False)

# Save classification report
report = classification_report(
    actual_test_labels,
    predicted_test_labels,
    labels=list(range(11)),
    target_names=class_names,
    output_dict=True,
    zero_division=0
)

pd.DataFrame(report).transpose().to_csv(
    results_dir / "bert_classification_report.csv"
)

# Save confusion matrix
cm = confusion_matrix(
    actual_test_labels,
    predicted_test_labels,
    labels=list(range(11))
)

pd.DataFrame(
    cm,
    index=class_names,
    columns=class_names
).to_csv(results_dir / "bert_confusion_matrix.csv")

print("Final BERT evaluation results saved to:", results_dir)

Final BERT evaluation results saved to: /Users/emilybartell/Documents/CS1684 Crisis NLP/cs1684-crisis-nlp/outputs/bert_evaluation
